### 💡 Apriori 알고리즘

`Apriori` 알고리즘이란 **거래 데이터에서 자주 등장하는 Frequent Itemsets(빈발 항목 집합)을 탐색**하는 알고리즘이다.

Apriori를 활용한 연관 규칙 분석은 크게 두 단계로 구성된다.
1. 빈발 항목 집합 탐색 및 Support 계산
2. 연관 규칙 생성 및 Confidence, Lift 계산

Apriori는 분류나 회기처럼 새로운 데이터의 결과를 예측하는 알고리즘이 아니라, 기존 데이터에 숨어 있는 패턴을 발견하는 알고리즘이다.

#### 1️⃣ Search Space (탐색 공간)
Search Space란 문제를 해결하기 위해 고려해야하는 모든 후보 조합의 집합을 말한다.
```python
items = ["우유", "빵", "계란"]
```
이라고 한다면 가능한 모든 후보 집합은 아래와 같다.

```text
# 총 7개
[우유] [빵] [계란] [우유, 빵] [우유, 계란] [빵, 계란] [우유, 빵, 계란]
```

상품이 3개일때는 7개밖에 안나오지만, 상품이 많아질수록 가능한 조합의 수가 급격하게 증가한다.

상품의 개수가 N개일때 가능한 집합의 수는 다음과 같다.

$2^N-1$

실제로는 수천, 수만개의 데이터가 존재할 수 있기 때문에 모든 조합을 계산하는건 매우 비효율적이다.

때문에, Apriori는 탐색해야 할 후보 자체를 줄이는 전략을 선택한다.

#### 2️⃣ Apriori의 원리

Apriori의 핵심 원리는 어떤 항목 집합이 최소 지지도를 만족하지 못하면, 해당 항목 집합을 포함하는 더 큰 항목 집합도 최소 지지도를 만족할 수 없다.

$Support(A,B,C)\leq Support(A,B)\leq Support(A)$

예를들어 최소 지지도를 0.3으로 설정했다고 가정할 때

만약 C 자체의 support가 0.1이라면 C를 포함한 모든 집합의 support는 0.1보다 낮을 것이기 때문에 c 자체를 제거해버린다.

이렇게 불필요한 후보를 제거하는 과정은 `Pruning`(가지치기)이라고 한다.

In [ ]:
import numpy as np
import pandas as pd

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

transactions = [
    ["노트북", "마우스", "키보드"],
    ["노트북", "마우스"],
    ["모니터", "키보드"],
    ["노트북", "모니터", "마우스"],
    ["마우스", "키보드"],
    ["노트북", "마우스", "키보드"],
    ["모니터", "마우스"],
    ["노트북", "키보드"],
    ["노트북", "모니터"],
    ["노트북", "마우스", "모니터", "키보드"]
]

encoder = TransactionEncoder()

encoded_data = encoder.fit_transform(transactions)

df = pd.DataFrame(
    encoded_data,
    columns=encoder.columns_
)

#### ☑️ Apriori()

In [ ]:
# apriori() : 빈발 항목 집합을 탐색하는 함수
# df : boolean 또는 0/1로 나타낸 데이터 프레임
# min_support : 최소 지지도
# use_colnames : 상품이름 사용여부
frequent_itemsets = apriori(
    df,
    min_support=0.4,
    use_colnames=True
)

# frozenset은 생성 이후 원소 추가/삭제가 불가능한 집합 자료구조이다.
print(frequent_itemsets)

   support               itemsets
0      0.7       frozenset({노트북})
1      0.7       frozenset({마우스})
2      0.5       frozenset({모니터})
3      0.6       frozenset({키보드})
4      0.5  frozenset({노트북, 마우스})
5      0.4  frozenset({키보드, 노트북})
6      0.4  frozenset({키보드, 마우스})


#### ☑️ association_rules()

In [16]:
# 위에서 만든 빈발 항목 집합을 기반으로
# Confidence가 0.5 이상인 연관 규칙을 생성한다.

# df : 빈발 항목 집합 데이터프레임
# metric="confidence" : 연관 규칙을 선택할 기준 지표. lift로도 설정이 가능하다.
# min_threshold=0.5 : 기준 지표의 최솟값
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5
)

# 여러가지 컬럼이 있는데 확인할 주요한 컬럼만 확인
columns = [
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]

print(rules[columns])

        antecedents       consequents  support  confidence      lift
0  frozenset({노트북})  frozenset({마우스})      0.5    0.714286  1.020408
1  frozenset({마우스})  frozenset({노트북})      0.5    0.714286  1.020408
2  frozenset({키보드})  frozenset({노트북})      0.4    0.666667  0.952381
3  frozenset({노트북})  frozenset({키보드})      0.4    0.571429  0.952381
4  frozenset({키보드})  frozenset({마우스})      0.4    0.666667  0.952381
5  frozenset({마우스})  frozenset({키보드})      0.4    0.571429  0.952381


In [18]:
# 원하는 조건을 적용해서 필요한 결과만 출력하기
filtered_rules = rules[
    (rules["support"] >= 0.4) &
    (rules["confidence"] >= 0.6) &
    (rules["lift"] > 1.0)
]

print(filtered_rules[columns])

        antecedents       consequents  support  confidence      lift
0  frozenset({노트북})  frozenset({마우스})      0.5    0.714286  1.020408
1  frozenset({마우스})  frozenset({노트북})      0.5    0.714286  1.020408


In [25]:
# Lift 기준 내림차순 정렬
sorted_rules = rules.sort_values(
    by="lift",
    ascending=False
)

print(sorted_rules[columns])

# lift가 높은 순서대로 정렬
# lift가 동일하다면 confidence가 높은 순서대로 정렬
# confidence가 동일하다면 support가 높은 순서대로 정렬
sorted_rules = rules.sort_values(
    by=["lift", "confidence", "support"],
    ascending=[False, False, False]
)

print(sorted_rules[columns])

        antecedents       consequents  support  confidence      lift
0  frozenset({노트북})  frozenset({마우스})      0.5    0.714286  1.020408
1  frozenset({마우스})  frozenset({노트북})      0.5    0.714286  1.020408
2  frozenset({키보드})  frozenset({노트북})      0.4    0.666667  0.952381
3  frozenset({노트북})  frozenset({키보드})      0.4    0.571429  0.952381
4  frozenset({키보드})  frozenset({마우스})      0.4    0.666667  0.952381
5  frozenset({마우스})  frozenset({키보드})      0.4    0.571429  0.952381
        antecedents       consequents  support  confidence      lift
0  frozenset({노트북})  frozenset({마우스})      0.5    0.714286  1.020408
1  frozenset({마우스})  frozenset({노트북})      0.5    0.714286  1.020408
2  frozenset({키보드})  frozenset({노트북})      0.4    0.666667  0.952381
4  frozenset({키보드})  frozenset({마우스})      0.4    0.666667  0.952381
3  frozenset({노트북})  frozenset({키보드})      0.4    0.571429  0.952381
5  frozenset({마우스})  frozenset({키보드})      0.4    0.571429  0.952381
